# LangChain Agent with Short-Term Memory
Short-term memory allows an agent to remember conversation history within a thread using a **checkpointer**.

In [ ]:
!pip install langchain langchain-openai langgraph

In [ ]:
from google.colab import userdata
import os
os.environ["AZURE_OPENAI_API_KEY"] = userdata.get('AZURE_OPENAI_API_KEY')
os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get('AZURE_OPENAI_ENDPOINT')
os.environ["OPENAI_API_VERSION"] = "2025-03-01-preview"

In [ ]:
from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    model="gpt-4.1-mini",
    azure_deployment="gpt-4.1-mini"
)

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

def get_current_time(city: str) -> dict:
    """Return the current time in a specified city."""
    return {"city": city, "time": "12:00 PM"}

agent = create_agent(
    model,
    tools=[get_current_time],
    system_prompt="You are a helpful assistant. Remember what the user tells you.",
    checkpointer=InMemorySaver(),
)

In [ ]:
# All invocations with the same thread_id share conversation history
thread = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi! My name is Raj and I live in Jamui."}]},
    thread,
)
print(response["messages"][-1].content)

In [ ]:
# Agent remembers previous messages in the same thread
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name and where do I live?"}]},
    thread,
)
print(response["messages"][-1].content)

In [ ]:
# Agent can also use tools while maintaining memory
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the time in my city?"}]},
    thread,
)
print(response["messages"][-1].content)

In [ ]:
# A different thread_id starts a fresh conversation (no memory of thread 1)
thread2 = {"configurable": {"thread_id": "2"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    thread2,
)
print(response["messages"][-1].content)